1) Using the attached dataset (mlb_pitch_velo_assessment.csv), build a model that predicts whether the first pitch of a baseball game by each starting pitcher will be faster than 89.95 mph.

In [1]:
import pandas as pd
import pprint
import numpy as np
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, log_loss, brier_score_loss
import statsmodels.api as sm
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Load the dataset
df = pd.read_csv('mlb_pitch_velo_assessment.csv')

print(f'Total Rows: {df.shape[0]}')
print(f'Total Columns: {df.shape[0]}')
print(df.dtypes)

nan_counts = df.isna().sum()

print("Missing values in each column:")

print(nan_counts)

Total Rows: 291684
Total Columns: 291684
pitch_id               int64
game_id                int64
season                 int64
date                  object
home_team_id           int64
home_team_name        object
away_team_id           int64
away_team_name        object
venue_id               int64
pitch_number           int64
pitcher_id           float64
pitcher_name          object
batter_id              int64
batter_name           object
pre_pitch_inning       int64
is_top_half            int64
pre_pitch_outs         int64
pre_pitch_balls        int64
pre_pitch_strikes      int64
pitch_type            object
release_speed        float64
dtype: object
Missing values in each column:
pitch_id               0
game_id                0
season                 0
date                   0
home_team_id           0
home_team_name         0
away_team_id           0
away_team_name         0
venue_id               0
pitch_number           0
pitcher_id             0
pitcher_name           0
batte

In [2]:
# Filtering data to only include first pitch for each game and pitcher
first_pitch = df[df['pitch_number'] == 1]

In [3]:
nan_counts_1 = first_pitch.isna().sum()

print("Missing values in each column for just the first pitch of the game:")

print(nan_counts_1)

Missing values in each column for just the first pitch:
pitch_id             0
game_id              0
season               0
date                 0
home_team_id         0
home_team_name       0
away_team_id         0
away_team_name       0
venue_id             0
pitch_number         0
pitcher_id           0
pitcher_name         0
batter_id            0
batter_name          0
pre_pitch_inning     0
is_top_half          0
pre_pitch_outs       0
pre_pitch_balls      0
pre_pitch_strikes    0
pitch_type           1
release_speed        1
dtype: int64


In [4]:
#Binary Target Variable - '1' if the pitch is over 89.95 mph and 0 if below
first_pitch['over_90'] = (first_pitch['release_speed'] > 89.95).astype(int)
label_encoder = LabelEncoder()
first_pitch['pitch_type'] = label_encoder.fit_transform(first_pitch['pitch_type'])

<ipython-input-4-ba7653432706>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_pitch['over_90'] = (first_pitch['release_speed'] > 89.95).astype(int)
<ipython-input-4-ba7653432706>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_pitch['pitch_type'] = label_encoder.fit_transform(first_pitch['pitch_type'])


In [5]:
first_pitch['pitcher_id'] = first_pitch['pitcher_id'].astype('Int64')

<ipython-input-5-bd7f636a1254>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_pitch['pitcher_id'] = first_pitch['pitcher_id'].astype('Int64')


In [6]:
first_pitch['pre_pitch_inning'].value_counts()
#pre_pitch_inning should only be 1

,count
pre_pitch_inning,
1,1630


In [7]:
first_pitch[['over_90','pitch_type','batter_id','venue_id','pitcher_id',
               'home_team_id','away_team_id']].corr()['over_90']

#seeing correlations with pitch speed over 89.95mph or not
#pitcher_id seems to be the most correlated with whether a pitch is over 89.95mph

,over_90
over_90,1.000000
pitch_type,-0.066847
batter_id,0.018315
venue_id,-0.060623
pitcher_id,0.313781
home_team_id,-0.008117
away_team_id,0.010303


In [8]:
model_types = ['logit', 'rf','xgboost']
models = {}
predictions = {}

for name in model_types:
  models[name] ={}
  predictions[name] = {}

In [9]:
# Define the features and target variable
features = ['pitch_type', 'batter_id', 'venue_id', 'pitcher_id', 'home_team_id', 'away_team_id']

# Remove rows with any missing values (NA) in the dataset for both features and target variable
first_pitch_cleaned = first_pitch.dropna(subset=features + ['over_90'])

In [10]:
# Ensure all data is numeric (for all feature columns and target)
first_pitch_cleaned[features] = first_pitch_cleaned[features].apply(pd.to_numeric, errors='coerce')
first_pitch_cleaned['over_90'] = pd.to_numeric(first_pitch_cleaned['over_90'], errors='coerce')

# X (features)
X = first_pitch_cleaned[features]

# y (target): The target variable is 'over_90'
y = first_pitch_cleaned['over_90']

# Split the data into training and testing sets (70% training, 30% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

X_train = X_train.astype('int64')
X_test = X_test.astype('int64')
y_train = y_train.astype('int64')
y_test = y_test.astype('int64')
#ensuring values are all numerical


In [11]:
def ModelOutputsFromModelAndFeatures(model, features):
    X_train_subset = X_train[features]
    X_test_subset = X_test[features]
    # Print the accuracy of the model on both training and test datasets
    print('Accuracy')
    print(f'Train : {model.score(X_train_subset, y_train):.3f}')
    print(f'Test : {model.score(X_test_subset, y_test):.3f}')
    model_predicted = model.predict(X_test_subset)
    #Generating predicted probabilities and predictions from the test set
    model_predicted_probabilities = model.predict_proba(X_test_subset)[:, 1]
    #printing evaluation metrics
    print(f'AUC : {roc_auc_score(y_test, model_predicted_probabilities):.3f}')
    print(f'Log Loss : {log_loss(y_test, model_predicted_probabilities):.3f}')
    print(f'Brier : {brier_score_loss(y_test, model_predicted_probabilities):.3f}')
    print(classification_report(y_test, model_predicted))

In [12]:
# Logistic Regression Model and Output
def LogisticRegressionAutoModel(features):
    X_train_subset = X_train[features]
    X_test_subset = X_test[features]
    # Initialize and fit Logistic Regression model
    logit = LogisticRegression(max_iter=1000)  # Increased max_iter to prevent convergence issues
    logit.fit(X_train_subset, y_train)
    # Print the intercept and coefficients
    print("Intercept : " + str(logit.intercept_))
    print("Coefs : " + str(logit.coef_))
    # Print accuracy and other metrics
    print('Accuracy')
    ModelOutputsFromModelAndFeatures(logit, features)
    # Add constant for statsmodels logistic regression model
    X_train2 = sm.add_constant(X_train_subset)
    # Ensure the data passed to statsmodels is numeric
    X_train2 = X_train2.apply(pd.to_numeric, errors='coerce')
    est = sm.Logit(y_train, X_train2).fit()
    print(est.summary()) #print model summary

    return logit


In [13]:
def RandomForestAutoModel(features):
    X_train_subset = X_train[features]
    X_test_subset = X_test[features]

    # Initialize and fit Random Forest model with hyperparameters to reduce overfitting
    rf = RandomForestClassifier(n_estimators=100, random_state=42,
                            max_depth=10, min_samples_split=10,
                            min_samples_leaf=5, max_features='sqrt',
                            bootstrap=True, class_weight='balanced')

    rf.fit(X_train_subset, y_train) #fitting model

    # Print feature importances
    print("Feature Importances : " + str(rf.feature_importances_))

    # Print accuracy and other metrics
    print('Accuracy')
    ModelOutputsFromModelAndFeatures(rf, features)

    return rf


In [14]:
def XGBoostAutoModel(features):
    X_train_subset = X_train[features]
    X_test_subset = X_test[features]

    # Initialize and fit XGBoost model with hyperparameters to reduce overfitting
    xgb = XGBClassifier(n_estimators=100, random_state=42,
                        max_depth=10, min_child_weight=5,
                        learning_rate=0.1, subsample=0.8,
                        colsample_bytree=0.8, scale_pos_weight=1,
                        use_label_encoder=False, eval_metric='logloss')

    # Fit the model on the training data
    xgb.fit(X_train_subset, y_train)

    # Print feature importances
    print("Feature Importances : " + str(xgb.feature_importances_))

    # Print accuracy and other metrics
    print('Accuracy')
    ModelOutputsFromModelAndFeatures(xgb, features)

    return xgb

In [15]:
def AccuracyMetricsUpdateByName(models, model_name):
  models[model_name]['auc'] = roc_auc_score(y_test, predictions[model_name]['predicted_probabilities'])
  models[model_name]['logloss'] = log_loss(y_test, predictions[model_name]['predicted_probabilities'])
  models[model_name]['brier'] = brier_score_loss(y_test, predictions[model_name]['predicted_probabilities'])
  return models

# **Logistic Regression Analysis**

In [16]:
logit = LogisticRegressionAutoModel(features)

Intercept : [0.725936]
Coefs : [[-2.59759917e-01  4.64104226e-05 -4.11032737e-03  4.34986490e-04
  -6.80324887e-03 -7.41570837e-03]]
Accuracy
Accuracy
Train : 0.803
Test : 0.755
AUC : 0.727
Log Loss : 0.485
Brier : 0.159
              precision    recall  f1-score   support

           0       0.26      0.09      0.13       104
           1       0.79      0.94      0.86       385

    accuracy                           0.75       489
   macro avg       0.53      0.51      0.49       489
weighted avg       0.68      0.75      0.70       489

Optimization terminated successfully.
         Current function value: 0.426126
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                over_90   No. Observations:                 1141
Model:                          Logit   Df Residuals:                     1134
Method:                           MLE   Df Model:                            6
Date:                Tue, 25 Mar 2

Let's now set a baseline model as it appears **pitch_type** and **pitcher_id** are the most important predictors (very low p-values). I will try to add each feature to  **pitch_type** and **pitcher_id** to see which other variables give it the most lift.

In [17]:
from re import L
logit_models = {}
for feature in features:
  print(feature)
  if feature == "pitch_type":
    logit = LogisticRegressionAutoModel(['pitch_type'])

    predicted_probabilities = logit.predict_proba(X_test[['pitch_type']])[:,1]

    logit_models['pitch_type'] = {}
    logit_models['pitch_type']['auc'] = roc_auc_score(y_test, predicted_probabilities)
    logit_models['pitch_type']['log_loss'] = log_loss(y_test, predicted_probabilities)
    logit_models['pitch_type']['brier'] = brier_score_loss(y_test, predicted_probabilities)

  else:
      logit = LogisticRegressionAutoModel(['pitch_type',feature])
      #creating model with pitch_type and other feature variable inserted
      predicted_probabilities = logit.predict_proba(X_test[['pitch_type',feature]])[:,1]

      if ('pitch_type_and_' + str(feature)) not in logit_models:
        logit_models['pitch_type_and_' + str(feature)] = {}
        logit_models['pitch_type_and_' + str(feature)]['auc'] = {}
        logit_models['pitch_type_and_' + str(feature)]['logloss'] = {}
        logit_models['pitch_type_and_' + str(feature)]['brier'] = {}

      logit_models['pitch_type_and_' + str(feature)]['auc'] = roc_auc_score(y_test, predicted_probabilities)
      logit_models['pitch_type_and_' + str(feature)]['logloss'] = log_loss(y_test, predicted_probabilities)
      logit_models['pitch_type_and_' + str(feature)]['brier'] = brier_score_loss

pprint.pprint(logit_models)


pitch_type
Intercept : [1.98279826]
Coefs : [[-0.14297995]]
Accuracy
Accuracy
Train : 0.814
Test : 0.787
AUC : 0.538
Log Loss : 0.516
Brier : 0.167
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       104
           1       0.79      1.00      0.88       385

    accuracy                           0.79       489
   macro avg       0.39      0.50      0.44       489
weighted avg       0.62      0.79      0.69       489

Optimization terminated successfully.
         Current function value: 0.478471
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                over_90   No. Observations:                 1141
Model:                          Logit   Df Residuals:                     1139
Method:                           MLE   Df Model:                            1
Date:                Tue, 25 Mar 2025   Pseudo R-squ.:                0.003349
Time:                       

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

                           Logit Regression Results                           
Dep. Variable:                over_90   No. Observations:                 1141
Model:                          Logit   Df Residuals:                     1138
Method:                           MLE   Df Model:                            2
Date:                Tue, 25 Mar 2025   Pseudo R-squ.:                0.004530
Time:                        12:22:48   Log-Likelihood:                -545.29
converged:                       True   LL-Null:                       -547.77
Covariance Type:            nonrobust   LLR p-value:                   0.08364
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7129      0.362      4.732      0.000       1.003       2.422
pitch_type    -0.1412      0.074     -1.898      0.058      -0.287       0.005
batter_id   5.343e-05   4.62e-05      1.156      0.2

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


pitcher_id
Intercept : [0.66878908]
Coefs : [[-0.26257467  0.00043438]]
Accuracy
Accuracy
Train : 0.814
Test : 0.775
AUC : 0.725
Log Loss : 0.482
Brier : 0.157
              precision    recall  f1-score   support

           0       0.38      0.09      0.14       104
           1       0.80      0.96      0.87       385

    accuracy                           0.78       489
   macro avg       0.59      0.52      0.51       489
weighted avg       0.71      0.78      0.72       489

Optimization terminated successfully.
         Current function value: 0.427249
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                over_90   No. Observations:                 1141
Model:                          Logit   Df Residuals:                     1138
Method:                           MLE   Df Model:                            2
Date:                Tue, 25 Mar 2025   Pseudo R-squ.:                  0.1100
Time:           

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

Based on the above models, in all likelihood we would only keep **pitch_type**  and **pitcher_id**. The other variables do not appear to provide much lift!

In [18]:
features1 = ['pitch_type','pitcher_id']
X_train_subset = X_train[features1]
X_test_subset = X_test[features1]

In [19]:
logit = LogisticRegressionAutoModel(features1)

Intercept : [0.66878908]
Coefs : [[-0.26257467  0.00043438]]
Accuracy
Accuracy
Train : 0.814
Test : 0.775
AUC : 0.725
Log Loss : 0.482
Brier : 0.157
              precision    recall  f1-score   support

           0       0.38      0.09      0.14       104
           1       0.80      0.96      0.87       385

    accuracy                           0.78       489
   macro avg       0.59      0.52      0.51       489
weighted avg       0.71      0.78      0.72       489

Optimization terminated successfully.
         Current function value: 0.427249
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                over_90   No. Observations:                 1141
Model:                          Logit   Df Residuals:                     1138
Method:                           MLE   Df Model:                            2
Date:                Tue, 25 Mar 2025   Pseudo R-squ.:                  0.1100
Time:                      

# **Random Forest Analysis**

In [20]:
rf = RandomForestAutoModel(features)

Feature Importances : [0.0967638  0.07132656 0.17700868 0.42487426 0.17563262 0.05439408]
Accuracy
Accuracy
Train : 0.958
Test : 0.918
AUC : 0.960
Log Loss : 0.240
Brier : 0.065
              precision    recall  f1-score   support

           0       0.80      0.83      0.81       104
           1       0.95      0.94      0.95       385

    accuracy                           0.92       489
   macro avg       0.87      0.88      0.88       489
weighted avg       0.92      0.92      0.92       489



Let's now set a baseline model as it appears **pitcher_id** is the most important predictor. I will try to add each feature to  **pitcher_id** to see which other variables give it the most lift.

In [21]:
from re import L
rf_models = {}
for feature in features:
  print(feature)
  if feature == "pitcher_id":
    rf = RandomForestAutoModel(['pitcher_id'])

    predicted_probabilities = rf.predict_proba(X_test[['pitcher_id']])[:,1]

    rf_models['pitcher_id'] = {}
    rf_models['pitcher_id']['auc'] = roc_auc_score(y_test, predicted_probabilities)
    rf_models['pitcher_id']['log_loss'] = log_loss(y_test, predicted_probabilities)
    rf_models['pitcher_id']['brier'] = brier_score_loss(y_test, predicted_probabilities)

  else:
      rf = RandomForestAutoModel(['pitcher_id',feature])

      predicted_probabilities = rf.predict_proba(X_test[['pitcher_id',feature]])[:,1]
      #creating model with pitcher_id and other feature variable inserted
      if ('pitcher_id_and_' + str(feature)) not in logit_models:
        rf_models['pitcher_id_and_' + str(feature)] = {}
        rf_models['pitcher_id_and_' + str(feature)]['auc'] = {}
        rf_models['pitcher_id_and_' + str(feature)]['logloss'] = {}
        rf_models['pitcher_id_and_' + str(feature)]['brier'] = {}

      rf_models['pitcher_id_and_' + str(feature)]['auc'] = roc_auc_score(y_test, predicted_probabilities)
      rf_models['pitcher_id_and_' + str(feature)]['logloss'] = log_loss(y_test, predicted_probabilities)
      rf_models['pitcher_id_and_' + str(feature)]['brier'] = brier_score_loss

pprint.pprint(rf_models)

pitch_type
Feature Importances : [0.86023458 0.13976542]
Accuracy
Accuracy
Train : 0.952
Test : 0.926
AUC : 0.963
Log Loss : 0.231
Brier : 0.066
              precision    recall  f1-score   support

           0       0.77      0.94      0.84       104
           1       0.98      0.92      0.95       385

    accuracy                           0.93       489
   macro avg       0.87      0.93      0.90       489
weighted avg       0.94      0.93      0.93       489

batter_id
Feature Importances : [0.76024684 0.23975316]
Accuracy
Accuracy
Train : 0.930
Test : 0.888
AUC : 0.907
Log Loss : 0.330
Brier : 0.097
              precision    recall  f1-score   support

           0       0.72      0.77      0.74       104
           1       0.94      0.92      0.93       385

    accuracy                           0.89       489
   macro avg       0.83      0.84      0.84       489
weighted avg       0.89      0.89      0.89       489

venue_id
Feature Importances : [0.65423965 0.34576035]
Ac

Based on the results above and similar to the Logisitc Regresison Model, the **pitcher_id** and **pitcher_type** combination is the best performing Random Forest Model.

In [22]:
features2 = ['pitch_type','pitcher_id']
X_train_subset = X_train[features2]
X_test_subset = X_test[features2]
rf = RandomForestAutoModel(features2)

Feature Importances : [0.13974583 0.86025417]
Accuracy
Accuracy
Train : 0.947
Test : 0.926
AUC : 0.962
Log Loss : 0.232
Brier : 0.066
              precision    recall  f1-score   support

           0       0.77      0.94      0.84       104
           1       0.98      0.92      0.95       385

    accuracy                           0.93       489
   macro avg       0.87      0.93      0.90       489
weighted avg       0.94      0.93      0.93       489



# **XGBoost** **Analysis**

In [23]:
xgboost = XGBoostAutoModel(features)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Feature Importances : [0.20177734 0.06268297 0.2279377  0.27055776 0.18647806 0.05056624]
Accuracy
Accuracy
Train : 0.962
Test : 0.910
AUC : 0.953
Log Loss : 0.196
Brier : 0.054
              precision    recall  f1-score   support

           0       0.84      0.71      0.77       104
           1       0.93      0.96      0.94       385

    accuracy                           0.91       489
   macro avg       0.88      0.84      0.86       489
weighted avg       0.91      0.91      0.91       489



Let's now set a baseline model as it appears **pitcher_id** is the most important predictor. I will try to add each feature to **pitcher_id** to see which other variables give it the most lift.

In [24]:
from re import L
xg_models = {}
for feature in features:
  print(feature)
  if feature == "pitcher_id":
    xgboost = XGBoostAutoModel(['pitcher_id'])

    predicted_probabilities = xgboost.predict_proba(X_test[['pitcher_id']])[:,1]
    #creating model with pitcher_id and other feature variable inserted

    xg_models['pitcher_id'] = {}
    xg_models['pitcher_id']['auc'] = roc_auc_score(y_test, predicted_probabilities)
    xg_models['pitcher_id']['log_loss'] = log_loss(y_test, predicted_probabilities)
    xg_models['pitcher_id']['brier'] = brier_score_loss(y_test, predicted_probabilities)

  else:
      xgboost = XGBoostAutoModel(['pitcher_id',feature])

      predicted_probabilities = xgboost.predict_proba(X_test[['pitcher_id',feature]])[:,1]

      if ('pitcher_id_and_' + str(feature)) not in logit_models:
        xg_models['pitcher_id_and_' + str(feature)] = {}
        xg_models['pitcher_id_and_' + str(feature)]['auc'] = {}
        xg_models['pitcher_id_and_' + str(feature)]['logloss'] = {}
        xg_models['pitcher_id_and_' + str(feature)]['brier'] = {}

      xg_models['pitcher_id_and_' + str(feature)]['auc'] = roc_auc_score(y_test, predicted_probabilities)
      xg_models['pitcher_id_and_' + str(feature)]['logloss'] = log_loss(y_test, predicted_probabilities)
      xg_models['pitcher_id_and_' + str(feature)]['brier'] = brier_score_loss

pprint.pprint(xg_models)

pitch_type
Feature Importances : [0.6056475 0.3943525]
Accuracy


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy
Train : 0.932
Test : 0.920
AUC : 0.950
Log Loss : 0.220
Brier : 0.061
              precision    recall  f1-score   support

           0       0.91      0.69      0.79       104
           1       0.92      0.98      0.95       385

    accuracy                           0.92       489
   macro avg       0.92      0.84      0.87       489
weighted avg       0.92      0.92      0.92       489

batter_id


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Feature Importances : [0.8670087  0.13299133]
Accuracy
Accuracy
Train : 0.945
Test : 0.902
AUC : 0.921
Log Loss : 0.256
Brier : 0.071
              precision    recall  f1-score   support

           0       0.85      0.65      0.74       104
           1       0.91      0.97      0.94       385

    accuracy                           0.90       489
   macro avg       0.88      0.81      0.84       489
weighted avg       0.90      0.90      0.90       489

venue_id
Feature Importances : [0.7152979  0.28470215]
Accuracy
Accuracy
Train : 0.937
Test : 0.918
AUC : 0.916
Log Loss : 0.239
Brier : 0.063
              precision    recall  f1-score   support

           0       0.86      0.73      0.79       104
           1       0.93      0.97      0.95       385

    accuracy                           0.92       489
   macro avg       0.90      0.85      0.87       489
weighted avg       0.92      0.92      0.92       489

pitcher_id
Feature Importances : [1.]
Accuracy
Accuracy
Train : 0.936

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


AUC : 0.926
Log Loss : 0.237
Brier : 0.065
              precision    recall  f1-score   support

           0       0.85      0.70      0.77       104
           1       0.92      0.97      0.94       385

    accuracy                           0.91       489
   macro avg       0.89      0.83      0.86       489
weighted avg       0.91      0.91      0.91       489

home_team_id
Feature Importances : [0.69055647 0.3094435 ]
Accuracy
Accuracy
Train : 0.928
Test : 0.910


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


AUC : 0.920
Log Loss : 0.240
Brier : 0.064
              precision    recall  f1-score   support

           0       0.92      0.63      0.75       104
           1       0.91      0.98      0.95       385

    accuracy                           0.91       489
   macro avg       0.91      0.81      0.85       489
weighted avg       0.91      0.91      0.90       489

away_team_id
Feature Importances : [0.9082817  0.09171832]
Accuracy
Accuracy
Train : 0.939
Test : 0.908
AUC : 0.921
Log Loss : 0.252
Brier : 0.070
              precision    recall  f1-score   support

           0       0.87      0.66      0.75       104
           1       0.91      0.97      0.94       385

    accuracy                           0.91       489
   macro avg       0.89      0.82      0.85       489
weighted avg       0.91      0.91      0.90       489

{'pitcher_id': {'auc': np.float64(0.9263236763236762),
                'brier': np.float64(0.0650232172947462),
                'log_loss': 0.23668105403927

Based on the results above and similar to the Logisitc Regresison and Random Forest Models, the **pitcher_id** and **pitcher_type** combination is the best XGBoost Model.

In [25]:
features3 = ['pitcher_id','pitch_type']
X_train_subset = X_train[features3]
X_test_subset = X_test[features3]
xgboost = XGBoostAutoModel(features3)

Feature Importances : [0.6056475 0.3943525]
Accuracy
Accuracy
Train : 0.932
Test : 0.920
AUC : 0.950
Log Loss : 0.220
Brier : 0.061
              precision    recall  f1-score   support

           0       0.91      0.69      0.79       104
           1       0.92      0.98      0.95       385

    accuracy                           0.92       489
   macro avg       0.92      0.84      0.87       489
weighted avg       0.92      0.92      0.92       489



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


## ***Full Analysis***

In [26]:
# Re-train with a consistent set of features
features_to_use = ['pitcher_id', 'pitch_type']

logit = LogisticRegressionAutoModel(features_to_use)
rf = RandomForestAutoModel(features_to_use)
xgboost = XGBoostAutoModel(features_to_use)

# Train the models with the selected features
logit.fit(X_train[features_to_use], y_train)
rf.fit(X_train[features_to_use], y_train)
xgboost.fit(X_train[features_to_use], y_train)

Intercept : [0.66878908]
Coefs : [[ 0.00043438 -0.26257467]]
Accuracy
Accuracy
Train : 0.814
Test : 0.775
AUC : 0.725
Log Loss : 0.482
Brier : 0.157
              precision    recall  f1-score   support

           0       0.38      0.09      0.14       104
           1       0.80      0.96      0.87       385

    accuracy                           0.78       489
   macro avg       0.59      0.52      0.51       489
weighted avg       0.71      0.78      0.72       489

Optimization terminated successfully.
         Current function value: 0.427249
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                over_90   No. Observations:                 1141
Model:                          Logit   Df Residuals:                     1138
Method:                           MLE   Df Model:                            2
Date:                Tue, 25 Mar 2025   Pseudo R-squ.:                  0.1100
Time:                      

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:22:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [27]:
predictions['logit']['predicted'] = logit.predict(X_test_subset)
predictions['logit']['predicted_probabilities'] = logit.predict_proba(X_test_subset)[:, 1]
predictions['rf']['predicted'] = rf.predict(X_test_subset)
predictions['rf']['predicted_probabilities'] = rf.predict_proba(X_test_subset)[:, 1]
predictions['xgboost']['predicted'] = xgboost.predict(X_test_subset)
predictions['xgboost']['predicted_probabilities'] = xgboost.predict_proba(X_test_subset)[:, 1]

AccuracyMetricsUpdateByName(models, 'logit')
AccuracyMetricsUpdateByName(models, 'rf')
AccuracyMetricsUpdateByName(models, 'xgboost')

{'logit': {'auc': np.float64(0.7253871128871129),
  'logloss': 0.4818824431089309,
  'brier': np.float64(0.15700749850229248)},
 'rf': {'auc': np.float64(0.9630244755244756),
  'logloss': 0.23097344929453303,
  'brier': np.float64(0.06579456397080832)},
 'xgboost': {'auc': np.float64(0.949975024975025),
  'logloss': 0.21990900200180624,
  'brier': np.float64(0.0608388787237596)}}

In conclusion, the XGBoost Model is the best choice among the models tested. The XGBoost and Random Forest Models outperformed the Logistic Regression Model in AUC, Log Loss, and Brier score. The XGBoost model performed the best in terms of the Log Loss and Brier score, indicating the XGBoost's high performance in minimizing prediction error and handling uncertainty in probability predictions. The Random Forest Model performed best in AUC, revealing its power to maximize model discriminative power. ***Moreover, the most reliable choice for the machine learning model is XGBoost because of its balanced performance across the 3 main metrics tested. ***